In [32]:
import pandas as pd

# Load the example submission file
example_submission = pd.read_csv("example.csv")

# Display the first few rows of the dataframe
example_submission.head()

# Load the template file
template = pd.read_csv("template.csv")

# Display the first few rows of the dataframe
template.head()


# Load the training data
training_data = pd.read_csv("training_data.csv")

# Display the first few rows of the dataframe
training_data.head()


# Load the testing data
testing_data = pd.read_csv("test_data.csv")

# Display the first few rows of the dataframe
testing_data.head()


,PatientID,Resp,PR Seq,RT Seq,VL-t0,CD4-t0
0,1,H,NCTCTATTAGATACAGGAGCAGATGACACAGTATTAGAAGARATGG...,CCTATTAGTCCTATTGAAACTGTACCAGTRAAATTAAAGCCAGGAA...,5.6,69
1,2,H,NCTCTATTAGATACAGGAGCAGATGATACAGTATTAGAAGAAATGA...,CCCATCAGTCCTATTGAAACTGTACCAGTAAAATTAAAGCCAGGAA...,5.3,119
2,3,H,GGGCAAATAAAGGAAGCTCTATTAGATACAGGAGCAGATGATACAG...,CCCATTAGTCCTATTGAAACTGTACCAGTAAAATTAAAGCCAGGAA...,5.7,41
3,4,H,GGGCAACTAAAGGAAGCTCTATTAGATACAGGAGCAGATGATACAG...,CCTATTAGTCCTATTGAAACTGTACCAGTAAAATTAAAGCCAGGAA...,5.2,48
4,5,H,GGGGGGCAACTAAAGGAAGCTCTATTAGATACAGGAGCAGATGATA...,CCCATTAGTCCTATTGAAACTGTACCAGTAAAATTAAAGCCAGGAA...,5.5,311


k-mer approach:

Error with missing data:


In [33]:
# Check for missing data in the PR Seq and RT Seq columns
pr_missing = training_data['PR Seq'].isna().sum()
rt_missing = training_data['RT Seq'].isna().sum()

pr_missing, rt_missing


(80, 0)

Impute:

In [34]:
# Compute the median length of the PR sequences
median_pr_len = training_data['PR Seq'].dropna().apply(len).median()

# Create a placeholder sequence of 'N's with the same length as the median sequence length
placeholder_seq = 'N' * int(median_pr_len)

# Fill the missing PR sequences with the placeholder sequence
training_data['PR Seq'].fillna(placeholder_seq, inplace=True)

# Check again for missing data in the PR Seq column
pr_missing = training_data['PR Seq'].isna().sum()
pr_missing


0

Simple model:

It seems there is a mismatch between the number of columns in the transformed counts data and the number of feature names provided by the vectorizer. This is likely due to the fact that not all possible di-nucleotides are present in both the PR and RT sequences, so when we fit the vectorizer on one set of sequences and transform the other, the sizes don't match.

A better approach would be to fit the vectorizer on the combined PR and RT sequences, so it learns all possible di-nucleotides. Then, we can transform the PR and RT sequences separately using this fitted vectorizer.

In [35]:
# Combine the PR and RT sequences
combined_seqs = pd.concat([training_data['PR Seq'], training_data['RT Seq'], testing_data['PR Seq'], testing_data['RT Seq']])

# Fit the vectorizer on the combined sequences
vectorizer.fit(combined_seqs)

# Transform the PR and RT sequences into di-nucleotide counts
pr_train_counts = vectorizer.transform(training_data['PR Seq'])
rt_train_counts = vectorizer.transform(training_data['RT Seq'])
pr_test_counts = vectorizer.transform(testing_data['PR Seq'])
rt_test_counts = vectorizer.transform(testing_data['RT Seq'])

# Convert the counts into dataframes
pr_train_df = pd.DataFrame(pr_train_counts.toarray(), columns=vectorizer.get_feature_names_out())
rt_train_df = pd.DataFrame(rt_train_counts.toarray(), columns=vectorizer.get_feature_names_out())
pr_test_df = pd.DataFrame(pr_test_counts.toarray(), columns=vectorizer.get_feature_names_out())
rt_test_df = pd.DataFrame(rt_test_counts.toarray(), columns=vectorizer.get_feature_names_out())

# Concatenate the PR and RT counts
train_counts_df = pd.concat([pr_train_df, rt_train_df], axis=1)
test_counts_df = pd.concat([pr_test_df, rt_test_df], axis=1)

# Add the PatientID, VL-t0, and CD4-t0 columns
train_counts_df['PatientID'] = training_data['PatientID']
train_counts_df['VL-t0'] = training_data['VL-t0']
train_counts_df['CD4-t0'] = training_data['CD4-t0']
test_counts_df['PatientID'] = testing_data['PatientID']
test_counts_df['VL-t0'] = testing_data['VL-t0']
test_counts_df['CD4-t0'] = testing_data['CD4-t0']

# Display the first few rows of the transformed training data
train_counts_df.head()


,aa,ab,ac,ad,ag,ah,ak,am,an,ar,...,ym,yn,yr,ys,yt,yw,yy,PatientID,VL-t0,CD4-t0
0,33,0,18,0,26,0,0,0,0,1,...,0,0,0,0,1,0,0,1,4.3,145
1,37,0,18,0,26,0,0,0,0,0,...,0,0,0,0,0,0,0,2,3.6,224
2,36,0,18,0,26,0,0,0,0,0,...,0,0,0,0,1,0,0,3,3.2,1017
3,33,0,17,0,28,0,0,0,0,0,...,0,0,0,0,0,0,0,4,5.7,206
4,34,0,18,0,28,0,0,0,0,0,...,0,0,0,0,0,0,0,5,3.5,572


Random Forest:

In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define the feature columns and the target variable
X = train_counts_df.drop(['PatientID'], axis=1)
y = training_data['Resp']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize a Random Forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred = rf.predict(X_val)

# Compute the accuracy of the model
accuracy = accuracy_score(y_val, y_val_pred)
accuracy


0.785

NN and XGBoost:

In [37]:
# Add prefixes to the column names
pr_train_df.columns = ['PR_' + col for col in pr_train_df.columns]
rt_train_df.columns = ['RT_' + col for col in rt_train_df.columns]
pr_test_df.columns = ['PR_' + col for col in pr_test_df.columns]
rt_test_df.columns = ['RT_' + col for col in rt_test_df.columns]

# Concatenate the PR and RT counts
train_counts_df = pd.concat([pr_train_df, rt_train_df], axis=1)
test_counts_df = pd.concat([pr_test_df, rt_test_df], axis=1)

# Add the PatientID, VL-t0, and CD4-t0 columns
train_counts_df['PatientID'] = training_data['PatientID']
train_counts_df['VL-t0'] = training_data['VL-t0']
train_counts_df['CD4-t0'] = training_data['CD4-t0']
test_counts_df['PatientID'] = testing_data['PatientID']
test_counts_df['VL-t0'] = testing_data['VL-t0']
test_counts_df['CD4-t0'] = testing_data['CD4-t0']

# Define the feature columns and the target variable
X = train_counts_df.drop(['PatientID'], axis=1)
y = training_data['Resp']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the MLP model
mlp.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred_mlp = mlp.predict(X_val)

# Compute the accuracy of the MLP model
accuracy_mlp = accuracy_score(y_val, y_val_pred_mlp)

# Train the XGBoost model
xgb.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred_xgb = xgb.predict(X_val)

# Compute the accuracy of the XGBoost model
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)

accuracy_mlp, accuracy_xgb


(0.785, 0.81)

Hyperparameter tuning xgBoost

Feature engineering:

In [38]:
# Function to calculate the GC content of a sequence
def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

# Add sequence length and GC content features for the PR and RT sequences
train_counts_df['PR_len'] = training_data['PR Seq'].apply(len)
train_counts_df['RT_len'] = training_data['RT Seq'].apply(len)
train_counts_df['PR_gc'] = training_data['PR Seq'].apply(gc_content)
train_counts_df['RT_gc'] = training_data['RT Seq'].apply(gc_content)
test_counts_df['PR_len'] = testing_data['PR Seq'].apply(len)
test_counts_df['RT_len'] = testing_data['RT Seq'].apply(len)
test_counts_df['PR_gc'] = testing_data['PR Seq'].apply(gc_content)
test_counts_df['RT_gc'] = testing_data['RT Seq'].apply(gc_content)

# Split the data into training and validation sets again
X = train_counts_df.drop(['PatientID'], axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the XGBoost model with the default parameters and the new features
xgb.fit(X_train, y_train)

# Make predictions on the validation set
y_val_pred_xgb = xgb.predict(X_val)

# Compute the accuracy of the XGBoost model
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)

accuracy_xgb


0.805

Create submission file predicting on test set:

In [39]:
# Make predictions on the test set
test_predictions = xgb.predict(test_counts_df.drop(['PatientID'], axis=1))

# Create a dataframe for the submission
submission_df = pd.DataFrame({
    'PatientID': test_counts_df['PatientID'],
    'Resp': test_predictions
})

# Display the submission dataframe
submission_df.head()


,PatientID,Resp
0,1,1
1,2,1
2,3,1
3,4,1
4,5,0


In [40]:
# Create a dataframe for the submission
submission_df = pd.DataFrame({
    'PatientID': template['PatientID'],
    'Resp': test_predictions
})

# Display the submission dataframe
submission_df.head()


,PatientID,Resp
0,1,1
1,2,1
2,3,1
3,4,1
4,5,0


In [41]:
# Save the submission dataframe to a CSV file
submission_df.to_csv('submission.csv', index=False)
